# Original H1: two T4 workers, FP16
RMS runs on GPU 0; Taper-minus on GPU 1. The 50-hour cap measures cumulative elapsed notebook time; worker hours are recorded separately. Data, effective batch, seeds, token budgets and decision thresholds are unchanged.

Attach **domain-shift-paired.zip** and your existing **stage1-online-inputs** private dataset. Start with `MODE='preflight'`. Later attach only the latest full paired preflight/resume archive. Single-GPU preflight and run archives cannot be used with this runner. Keep all historical full archives for saved weight snapshots. Do not attach reserved test data.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil, tempfile, time
inputs = Path('/kaggle/input')
roots = sorted({p.parent.parent.parent for p in inputs.rglob('paired.py')
                if p.parent.name == 'domain_shift_forgetting' and p.parent.parent.name == 'src'})
if roots:
    assert len(roots) == 1, f'Attach exactly one stage1 code bundle: {roots}'
    code = roots[0]
else:
    bundles = list(inputs.rglob('domain-shift-paired.zip'))
    assert len(bundles) == 1, 'Attach domain-shift-paired.zip as a private dataset'
    code = Path(tempfile.mkdtemp(prefix='stage1-code-'))
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist():
            assert (code / name).resolve().is_relative_to(code.resolve())
        z.extractall(code)
env = os.environ | {'PYTHONPATH': str(code / 'src'), 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
def cli(module, *args):
    import signal
    command = [sys.executable, '-m', 'domain_shift_forgetting.' + module, *map(str, args)]
    process = subprocess.Popen(command, env=env)
    try:
        result = process.wait()
    except KeyboardInterrupt:
        # Give the coordinator time to stop and join BOTH workers before export.
        if process.poll() is None:
            try:
                process.send_signal(signal.SIGTERM)
            except ProcessLookupError:
                pass
        try:
            process.wait(timeout=300)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
            print('Coordinator did not close safely. Full export will refuse active writers; preserve prior durable archive.')
        raise
    if result:
        raise subprocess.CalledProcessError(result, command)
print('Code:', code)


In [ ]:
# The CPU coordinator starts two subprocesses, each seeing exactly its assigned GPU.
env['CUDA_VISIBLE_DEVICES'] = ''
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', 'datasketch'], check=True)
MODE = 'preflight'  # then 'start', then 'resume'
SESSION_REMAINING_MINUTES = None
WEEKLY_GPU_HOURS_REMAINING = None
EXTERNAL_NOTEBOOK_HOURS = None  # cumulative experiment GPU-session time outside the coordinator ledger
RUN_PRIMARY = False
RECOVER_UNCLEAN = False  # only after confirming the previous session/worker ended
assert MODE in ('preflight', 'start', 'resume')
assert all(isinstance(x, (int, float)) and x >= 0 for x in
           (SESSION_REMAINING_MINUTES, WEEKLY_GPU_HOURS_REMAINING, EXTERNAL_NOTEBOOK_HOURS)), 'Fill all three usage values.'
settings_time = time.monotonic()
remaining_at_entry = min(SESSION_REMAINING_MINUTES, WEEKLY_GPU_HOURS_REMAINING * 60)
def remaining():
    value = remaining_at_entry - (time.monotonic() - settings_time) / 60
    assert value > 10, 'Insufficient time for a safe chunk and export reserve.'
    return value
scratch = Path(tempfile.mkdtemp(prefix='paired-'))
policy = code / 'configs/kaggle-stage1-paired.json'
online_roots = [p.parent for p in inputs.rglob('manifest.json') if p.parent.name == 'online'
                and json.loads(p.read_text()).get('schema') == 'pilot-arrays-v1']
if not online_roots:
    bundles = list(inputs.rglob('stage1-online-inputs.zip'))
    assert len(bundles) == 1, 'Attach your existing online inputs.'
    target = scratch / 'inputs'
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist():
            assert (target / name).resolve().is_relative_to(target.resolve())
        z.extractall(target)
    online_roots = [target / 'online']
assert len(online_roots) == 1, 'Attach exactly one online corpus.'
data = online_roots[0]
orders, upstream = data.parent / 'orders', data.parent / 'upstream'
root = scratch / 'run'
print('Mode:', MODE, 'data:', data, 'temporary run:', root)
print('Policy:', json.loads(policy.read_text())['protocol'], '| cap: 50 elapsed notebook hours')


In [ ]:
if MODE == 'preflight':
    preflight = scratch / 'paired-preflight'
    try:
        cli('paired', 'preflight', '--root', preflight, '--data', data, '--orders', orders,
            '--config', policy, '--tests', code / 'tests', '--remaining-minutes', remaining(),
            '--external-notebook-hours', EXTERNAL_NOTEBOOK_HOURS + (time.monotonic() - settings_time) / 3600)
    finally:
        stamp = str(int(time.time()))
        if preflight.exists():
            cli('paired', 'export', '--root', preflight, '--output', f'/kaggle/working/paired-preflight-{stamp}.zip')
            cli('paired', 'export', '--root', preflight, '--reports-only', '--output', f'/kaggle/working/paired-preflight-reports-{stamp}.zip')
    result = json.loads((preflight / 'paired-preflight.json').read_text())
    print('Paired preflight:', result['status'])
    if result['status'] == 'passed':
        print('Persist the FULL paired preflight archive. Start a NEW SESSION with it attached, MODE=start.')
    else:
        print('STOP: save the reports for review; do not start primary training.')


In [ ]:
if MODE == 'start':
    assert RUN_PRIMARY, 'Set RUN_PRIMARY=True after reviewing the paired preflight.'
    evidence = list(inputs.rglob('paired-preflight.json'))
    if evidence:
        assert len(evidence) == 1, 'Attach exactly one full paired preflight.'
        preflight = evidence[0].parent
    else:
        archives = [p for p in inputs.rglob('paired-preflight-*.zip') if 'reports' not in p.name]
        assert len(archives) == 1, 'Attach the full paired preflight archive.'
        preflight = scratch / 'imported-preflight'
        cli('paired', 'import', '--archive', archives[0], '--root', preflight)
    cli('paired', 'start', '--root', root, '--preflight', preflight, '--data', data, '--orders', orders,
        '--sources', upstream, '--config', policy, '--remaining-minutes', remaining(),
        '--external-notebook-hours', EXTERNAL_NOTEBOOK_HOURS)
elif MODE == 'resume':
    assert RUN_PRIMARY, 'Set RUN_PRIMARY=True to continue the same frozen matrix.'
    archives = [p for p in inputs.rglob('paired-resume-*.zip') if 'reports' not in p.name]
    if archives:
        assert len(archives) == 1, 'Attach only the latest full paired resume archive.'
        source = archives[0]
    else:
        receipts = [p for p in inputs.rglob('archive-receipt.json') if (p.parent / 'freeze.json').exists()
                    and json.loads((p.parent / 'freeze.json').read_text()).get('layout') == 'paired-v1']
        assert len(receipts) == 1, 'Attach the latest full paired resume archive.'
        source = receipts[0].parent
    cli('paired', 'import', '--archive', source, '--root', root)


In [ ]:
if MODE in ('start', 'resume'):
    # Account for this invocation's import, verification and freeze overhead before starting the ledger.
    external_at_run = EXTERNAL_NOTEBOOK_HOURS + (time.monotonic() - settings_time) / 3600
    try:
        cli('paired', 'run', '--root', root, '--data', data, '--orders', orders,
            '--remaining-minutes', remaining(), '--external-notebook-hours', external_at_run,
            '--recover-unclean', int(RECOVER_UNCLEAN))
    finally:
        stamp = str(int(time.time()))
        if (root / 'freeze.json').exists():
            try:
                cli('paired', 'report', '--root', root)
            finally:
                cli('paired', 'export', '--root', root, '--output', f'/kaggle/working/paired-resume-{stamp}.zip')
                cli('paired', 'export', '--root', root, '--reports-only', '--output', f'/kaggle/working/paired-reports-{stamp}.zip')
    print((root / 'status.json').read_text())
    print((root / 'decision-report.txt').read_text())
    ledger = json.loads((root / 'notebook-ledger.json').read_text())
    used = ledger['external_gpu_hours'] + sum(s['charged_seconds'] for s in ledger['sessions']) / 3600
    worker_hours = sum(s.get('summed_worker_seconds', 0) for s in ledger['sessions']) / 3600
    print(f'Cumulative experiment notebook hours: {used:.3f} / 50; recorded worker hours: {worker_hours:.3f}')
    print('External hours recorded at run start:', external_at_run)
    print('Save the FULL paired archive before stopping. Next session: MODE=resume. Keep all older archives.')


After every chunk save the complete paired-resume archive outside the session, then stop the session. Upload it privately and attach it next time. Both workers are joined before export; an expected prefix-persistence pause stops its peer safely. A failure pauses the experiment for investigation, never substitutes a seed. `RECOVER_UNCLEAN` does not bypass failures or corruption.

`EXTERNAL_NOTEBOOK_HOURS` counts experiment GPU-session time outside the parent coordinator ledger. Count overlapping two-GPU time once. Include previous single-GPU preflights, setup/idle/export, failed preflights, and restore time. This notebook automatically adds setup since the settings cell before each training invocation; do not add that same period twice next time. Add export and later idle time to the displayed external value for your next invocation. Neither external usage nor `notebook-ledger.json` resets at a weekly quota reset.
